In [1]:
%pip install -q "langchain==1.4.0" "langchain-text-splitters==1.1.2"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>로더는 원문과 출처 정보를 <code>Document</code> 객체로 묶습니다. 스플리터는 문서를 검색 단위인 청크로 나누며 메타데이터를 각 청크에 복사합니다.</p>

In [2]:
from importlib.metadata import version

print('langchain', version('langchain'))
print('langchain-text-splitters', version('langchain-text-splitters'))


langchain 1.4.0
langchain-text-splitters 1.1.2


<h3>예제 실행</h3>
<p>짧은 문장과 출처 메타데이터를 하나의 <code>Document</code> 객체로 만듭니다.</p>

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = '파란도서관은 월요일에 쉽니다. 화요일부터 일요일까지 운영합니다.'
docs = [Document(page_content=text, metadata={'doc_id': 'D01', 'page': 1})]
print(docs[0])


page_content='파란도서관은 월요일에 쉽니다. 화요일부터 일요일까지 운영합니다.' metadata={'doc_id': 'D01', 'page': 1}


<h3>분할 결과 확인</h3>
<p>문서를 나눈 뒤에도 각 청크에 출처 정보가 남는지 확인합니다.</p>

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=24, chunk_overlap=4)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    print(chunk.page_content, chunk.metadata)


파란도서관은 월요일에 쉽니다. 화요일부터 {'doc_id': 'D01', 'page': 1}
일요일까지 운영합니다. {'doc_id': 'D01', 'page': 1}


<h3>변형 실행</h3>
<p><code>chunk_size</code>를 18로 바꿨을 때 청크 수가 어떻게 달라지는지 확인합니다.</p>

In [5]:
smaller = RecursiveCharacterTextSplitter(chunk_size=18, chunk_overlap=4)
small_chunks = smaller.split_documents(docs)
print('청크 수:', len(small_chunks))


청크 수: 3


<h3>일부러 실패해 보기</h3>
<p>겹침 길이가 청크 크기보다 크면 분할 기준이 성립하지 않습니다. 라이브러리가 실제로 발생시키는 오류를 포착합니다.</p>

In [6]:
try:
    RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=11)
except ValueError as error:
    print('예상한 실패:', error)


예상한 실패: Got a larger chunk overlap (11) than chunk size (10), should be smaller.


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [7]:
assert chunks and all(c.metadata['doc_id'] == 'D01' for c in chunks)
assert len(small_chunks) >= len(chunks)
print('검증 통과: 내용과 출처 정보가 함께 분할되었습니다.')


검증 통과: 내용과 출처 정보가 함께 분할되었습니다.


<h3>정리</h3>
<p>검색 결과를 출처와 연결하려면 청크를 만들기 전에 문서 ID와 쪽 번호를 기록해야 합니다. 청크 크기와 겹침은 다음 실험에서 바꿀 설정입니다.</p>